### Text preprocess

In [1]:
import nltk
from nltk import tokenize
import numpy as np 
from string import punctuation
import unidecode
#stemmer = nltk.RSLPStemmer()


def proccess_text(tweets):
    
    # Removing links, mentions and hashtags
    tweets['processed_text'] = tweets.text.str.replace(r'(http\S+)', '',regex=True) \
                                          .str.replace(r'@[\w]*', '',regex=True) \
                                          .str.replace(r'#[\w]*','',regex=True) 
    print('[ok] - Removing links.')
    print('[ok] - Removing mentions.')
    print('[ok] - Removing hashtags.')

    textWords = ' '.join([text for text in tweets.processed_text])

    # Removing accent
    textWords = [unidecode.unidecode(text) for text in tweets.processed_text ]    
    print('[ok] - Removing accent.')
    
    # Creating a list of words and characters (stopwords) to be removed from the text
    # stopWords = nltk.corpus.stopwords.words("portuguese")    
    print('[ok] - Creating a list of words and characters (stopwords) to be removed from the text.')
    
    
    # Separating punctuation from words
    punctSeparator = tokenize.WordPunctTokenizer()
    punctuationList = list()
    for punct in punctuation:
        punctuationList.append(punct)
        
    #stopWords =   punctuationList + stopWords    
    stopWords =   punctuationList
    #print('[ok] - Separating punctuation from words.')


    # Iterating over the text and removing stop words 
    trasnformedText = list()    
    for text in textWords:
        newText = list()   
        text = text.lower()
        textWords = punctSeparator.tokenize(text)
        for words in textWords:
             if words not in stopWords:
                #newText.append(stemmer.stem(words))
                newText.append(words)
        trasnformedText.append(' '.join(newText))
    tweets.processed_text = trasnformedText
    print('[ok] - Removing punctuation and set text to lowecase.')
   
    # Removing all non-text characters
    tweets.processed_text = tweets['processed_text'].str.replace(r"[^a-zA-Z#]", " ", regex=True)                                                         
    print('[ok] - Removing all non-text characters.')
   
    trasnformedText = list()
    for phrase in tweets.processed_text:
        newPhrase = list()   
        newPhrase.append(' '.join(phrase.split()))
        for words in newPhrase:
            trasnformedText.append(''.join(newPhrase))
    tweets.processed_text = trasnformedText
    
    # Removing tweets with less than three terms
    index=[x for x in tweets.index if tweets.processed_text[x].count(' ') < 3]
    tweets = tweets.drop(index)
    print('[ok] - Removing tweets with less than three terms.')

    # Removing empty lines
    removeEmpty  = tweets.processed_text != ' '
    tweets = tweets[removeEmpty]
    print('[ok] - Removing empty lines.')

    tweets.reset_index(inplace=True)
    tweets = {'created_at': tweets.created_at, 'id':tweets.id,'author_id':tweets.author_id,'in_reply_to_user_id':tweets.in_reply_to_user_id, 'text': tweets.processed_text}
    #tweets = {'text': tweets.processed_text,'stance':tweets.stance}
    tweets = pd.DataFrame(tweets)
    tweets = tweets.sort_values(['created_at']).reset_index().drop(columns=["index"])
    #tweets = tweets.reset_index().drop(columns=["index"])
    
    return tweets

In [5]:
import pandas as pd 

docs = pd.read_csv('../data/collect/trump_tweets.csv')
docs

,created_at,id,author_id,in_reply_to_user_id,text
0,2016-01-01T00:03:29.000Z,682713720368140290,724599002,NaN,actually the biggest loser in
1,2016-01-01T00:22:51.000Z,682718591821721600,395484842,25073877.0,is their some point to s life
2,2016-01-01T00:23:39.000Z,682718793785815040,113457921,18916432.0,u sold out in looking forward to
3,2016-01-01T00:34:18.000Z,682721474566696960,51623437,NaN,get the picture happy new years
4,2016-01-01T00:39:18.000Z,682722734124711937,269651436,NaN,chicago for new years wouldn t be complete wit...
...,...,...,...,...,...
119263,2016-12-30T23:17:26.000Z,814973700923682820,237509961,NaN,tweet f ks putin says can t wait till he gets ...
119264,2016-12-30T23:23:38.000Z,814975259912839168,237509961,NaN,finds banana splits tribute band to headline i...
119265,2016-12-30T23:54:53.000Z,814983125411344384,3092209045,NaN,they are followers without followers evil cann...
119266,2016-12-30T23:57:19.000Z,814983738518110208,16362541,759251.0,is now selling a book about lmao at this crazi...


In [4]:
tweets = proccess_text(docs)

[ok] - Removing links.
[ok] - Removing mentions.
[ok] - Removing hashtags.
[ok] - Removing accent.
[ok] - Creating a list of words and characters (stopwords) to be removed from the text.
[ok] - Removing punctuation and set text to lowecase.
[ok] - Removing all non-text characters.
[ok] - Removing tweets with less than three terms.
[ok] - Removing empty lines.


In [7]:
tweets.to_csv('../data/teste.csv', index=False)